# Qwen PDF Fine-Tuning Comparison Notebook

This notebook fine-tunes Qwen on PDF research papers and then pushes:
1. LoRA adapter
2. Merged model
3. Quantized 4-bit model

It follows the same Hugging Face login pattern used in your earlier notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install -U  "transformers>=4.44.0" \
  "datasets>=2.20.0" \
  "peft>=0.12.0" \
  "accelerate>=0.34.0" \
  "bitsandbytes>=0.46.1" \
  "trl>=0.10.0" \
  "huggingface_hub>=0.23.0" \
  "sentencepiece>=0.2.0" \
  "pymupdf>=1.24.0" \
  "pypdf>=4.2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import glob
import random
from dataclasses import dataclass
from typing import List

import torch
from datasets import Dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4


In [ ]:
# Hugging Face authentication (same setup as your earlier notebook)
from google.colab import userdata

HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in from HF_TOKEN env var")
else:
    hf_token_secret = userdata.get("HF_TOKEN")
    if hf_token_secret:
        login(token=hf_token_secret)
        print("Logged in from HF_TOKEN secret var")
    else:
        print("HF_TOKEN not found. Run login() manually.")

Logged in from HF_TOKEN secret var


In [ ]:
@dataclass
class Config:
    # Keep Qwen for a fair comparison with your first approach
    base_model: str = "Qwen/Qwen2.5-7B-Instruct"

    # Put PDFs in /content/pdfs in Colab
    pdf_glob: str = "/content/drive/MyDrive/eval/papers/*.pdf"

    output_dir: str = "./qwen2_5_7b_pdf_lora"
    push_repo_id_adapter: str = "dizza01/qwen2.5-7b-pdf-lora"
    push_repo_id_merged: str = "dizza01/qwen2.5-7b-pdf-merged"
    push_repo_id_merged_4bit: str = "dizza01/qwen2.5-7b-pdf-merged-4bit"

    max_chars_per_doc: int = 500000
    chunk_size_chars: int = 2200
    chunk_overlap_chars: int = 300
    min_chunk_chars: int = 300
    test_size: float = 0.05

    # Keep abstain examples optional so we can isolate pure paper-finetuning effects first
    include_abstain_examples: bool = False
    abstain_sampling_ratio: float = 0.0

    max_seq_len: int = 1024

    # Step 3: lower adaptation strength to reduce drift from base instruct behavior
    epochs: float = 1.0
    learning_rate: float = 5e-5
    train_batch_size: int = 2
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0

    lora_r: int = 32
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    # Optional experiment tracking
    use_wandb: bool = False
    wandb_project: str = "bib-qwen-pdf-ft"
    wandb_run_name: str = "qwen25-7b-pdf-step3"

    early_stopping_patience: int = 2
    use_4bit: bool = True

cfg = Config()
cfg

Config(base_model='Qwen/Qwen2.5-7B-Instruct', pdf_glob='/content/drive/MyDrive/eval/papers/*.pdf', output_dir='./qwen2_5_7b_pdf_lora', push_repo_id_adapter='dizza01/qwen2.5-7b-pdf-lora', push_repo_id_merged='dizza01/qwen2.5-7b-pdf-merged', push_repo_id_merged_4bit='dizza01/qwen2.5-7b-pdf-merged-4bit', max_chars_per_doc=500000, chunk_size_chars=2200, chunk_overlap_chars=300, min_chunk_chars=300, test_size=0.05, max_seq_len=1024, epochs=1.5, learning_rate=0.0002, train_batch_size=2, eval_batch_size=2, grad_accum_steps=8, warmup_ratio=0.03, weight_decay=0.0, lora_r=64, lora_alpha=16, lora_dropout=0.05, use_4bit=True)

In [ ]:
# Upload PDFs to /content/pdfs before running this cell
os.makedirs("/content/drive/MyDrive/eval/papers", exist_ok=True)
pdf_files = sorted(glob.glob(cfg.pdf_glob))
print(f"Found {len(pdf_files)} PDFs")
for p in pdf_files[:10]:
    print(" -", p)

if len(pdf_files) == 0:
    raise ValueError("No PDFs found. Upload files to /content/pdfs first.")

Found 174 PDFs
 - /content/drive/MyDrive/eval/papers/4-Hydroxyglutamate_is_a_novel_predictor_of_pre-eclampsia_2020.pdf
 - /content/drive/MyDrive/eval/papers/A_comparison_of_South_Asian_specific_and_established_BMI_thresholds_for_determining_obesity_prevalence_in_pregnancy_and__2014.pdf
 - /content/drive/MyDrive/eval/papers/A_culturally_tailored_personaliseD_nutrition_intErvention_in_South_ASIan_women_at_risk_of_Gestational_Diabetes_Mellitus__2023.pdf
 - /content/drive/MyDrive/eval/papers/A_maternal_serum_metabolite_ratio_predicts_fetal_growth_restriction_at_term_2020.pdf
 - /content/drive/MyDrive/eval/papers/A_qualitative_study_in_parental_perceptions_and_understanding_of_SIDS-reduction_guidance_in_a_UK_bi-cultural_urban_commu_2016.pdf
 - /content/drive/MyDrive/eval/papers/A_quasi-experimental_effectiveness_evaluation_of_the_Incredible_Years_Toddler_parenting_programme_on_children_s_developm_2023.pdf
 - /content/drive/MyDrive/eval/papers/Agreement between parent reported and clinical c

In [ ]:
ABSTAIN_TEXT = "I cannot answer from the provided context."
SYSTEM_MSG = (
    "You are a QA assistant. Use only the provided context. "
    "If the answer is not in context, reply exactly: "
    f"{ABSTAIN_TEXT}"
)

def clean_text(t: str) -> str:
    t = t.replace("\x00", " ")
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def extract_pdf_text(path: str) -> str:
    text_parts: List[str] = []
    try:
        import fitz
        doc = fitz.open(path)
        for page in doc:
            text_parts.append(page.get_text("text"))
        doc.close()
    except Exception:
        from pypdf import PdfReader
        reader = PdfReader(path)
        for page in reader.pages:
            text_parts.append(page.extract_text() or "")
    return clean_text("\n".join(text_parts))

def chunk_text(text: str, size: int, overlap: int, min_len: int):
    chunks = []
    start = 0
    step = max(1, size - overlap)
    while start < len(text):
        c = text[start:start + size].strip()
        if len(c) >= min_len:
            chunks.append(c)
        start += step
    return chunks

def split_sentences(text: str):
    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if len(s.strip()) >= 40]
    return sents

def build_answerable_example(context: str, answer_sent: str):
    anchor = " ".join(answer_sent.split()[:10])
    question = f"According to the context, what is stated about: {anchor}?"
    answer = answer_sent
    text = (
        f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n"
        f"<|im_start|>user\nContext:\n{context}\n\nQuestion: {question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>\n"
    )
    return {"text": text, "label_type": "answerable"}

def build_abstain_example(context: str):
    # Optional abstain data (disabled by default via cfg.include_abstain_examples=False)
    question = "Based on the context, what dose-response effect is reported for a treatment not described in this excerpt?"
    answer = ABSTAIN_TEXT
    text = (
        f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n"
        f"<|im_start|>user\nContext:\n{context}\n\nQuestion: {question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>\n"
    )
    return {"text": text, "label_type": "abstain"}

raw_docs = []
for p in pdf_files:
    txt = extract_pdf_text(p)[:cfg.max_chars_per_doc]
    if txt:
        raw_docs.append({"source": os.path.basename(p), "text": txt})

records = []
abstain_ratio = min(1.0, max(0.0, float(getattr(cfg, "abstain_sampling_ratio", 0.0))))
include_abstain = bool(getattr(cfg, "include_abstain_examples", False)) and abstain_ratio > 0.0

for d in raw_docs:
    chunks = chunk_text(d["text"], cfg.chunk_size_chars, cfg.chunk_overlap_chars, cfg.min_chunk_chars)
    for c in chunks:
        sents = split_sentences(c)
        if not sents:
            continue

        answer_sent = random.choice(sents[: min(6, len(sents))])
        records.append(build_answerable_example(c, answer_sent))

        if include_abstain and random.random() < abstain_ratio:
            records.append(build_abstain_example(c))

print(f"Parsed docs: {len(raw_docs)}")
print(f"Training records: {len(records)}")
print(f"Abstain examples enabled: {include_abstain} (ratio={abstain_ratio})")

dataset = Dataset.from_list(records)
split = dataset.train_test_split(test_size=cfg.test_size, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds, eval_ds)
print("Sample label counts:", dataset.to_pandas()["label_type"].value_counts().to_dict())

Parsed docs: 174
Training records: 5419
Dataset({
    features: ['text'],
    num_rows: 5148
}) Dataset({
    features: ['text'],
    num_rows: 271
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

assistant_start_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
assistant_end_ids = tokenizer.encode("<|im_end|>", add_special_tokens=False)

def _find_subseq(seq, subseq, start_idx=0):
    if not subseq:
        return -1
    last = len(seq) - len(subseq) + 1
    for i in range(max(0, start_idx), max(0, last)):
        if seq[i : i + len(subseq)] == subseq:
            return i
    return -1

def tokenize_batch(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg.max_seq_len,
    )

    labels = []

    for input_ids in out["input_ids"]:
        row_labels = [-100] * len(input_ids)

        start = _find_subseq(input_ids, assistant_start_ids, 0)
        if start != -1:
            answer_start = start + len(assistant_start_ids)
            end = _find_subseq(input_ids, assistant_end_ids, answer_start)
            answer_end = end if end != -1 else len(input_ids)

            for j in range(answer_start, answer_end):
                token_id = input_ids[j]
                if token_id != tokenizer.pad_token_id:
                    row_labels[j] = token_id

        labels.append(row_labels)

    out["labels"] = labels
    return out

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_batch, batched=True, remove_columns=eval_ds.column_names)
train_tok.set_format(type="torch")
eval_tok.set_format(type="torch")

def _count_supervised_tokens(ds):
    return sum(sum(1 for tok in row if tok != -100) for row in ds["labels"])

train_supervised = _count_supervised_tokens(train_tok)
eval_supervised = _count_supervised_tokens(eval_tok)
print(f"Assistant-only supervised tokens -> train: {train_supervised}, eval: {eval_supervised}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5148 [00:00<?, ? examples/s]

Map:   0%|          | 0/271 [00:00<?, ? examples/s]

In [ ]:
if cfg.use_4bit and not torch.cuda.is_available():
    raise RuntimeError("use_4bit=True requires CUDA.")

model_kwargs = {"trust_remote_code": True}
if cfg.use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(cfg.base_model, **model_kwargs)
model.config.use_cache = False
if cfg.use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 161,480,704 || all params: 7,777,097,216 || trainable%: 2.0764


In [ ]:
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

if cfg.use_wandb:
    os.environ.setdefault("WANDB_PROJECT", cfg.wandb_project)

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="cosine",
    bf16=bf16,
    fp16=fp16,
    gradient_checkpointing=True,
    report_to=["wandb"] if cfg.use_wandb else ["none"],
    run_name=cfg.wandb_run_name if cfg.use_wandb else None,
    dataloader_pin_memory=False,
)

callbacks = []
if cfg.early_stopping_patience > 0:
    callbacks.append(
        EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience)
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    # Preserve precomputed assistant-only labels as-is.
    data_collator=default_data_collator,
    callbacks=callbacks,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
100,1.598233,1.676534
200,1.596033,1.630259
300,1.565969,1.605464
400,1.470757,1.594229
483,1.454161,1.592945


TrainOutput(global_step=483, training_loss=1.5802666030315138, metrics={'train_runtime': 16597.947, 'train_samples_per_second': 0.465, 'train_steps_per_second': 0.029, 'total_flos': 3.4320838142459904e+17, 'train_loss': 1.5802666030315138, 'epoch': 1.5003885003885005})

In [ ]:
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

model.push_to_hub(cfg.push_repo_id_adapter)
tokenizer.push_to_hub(cfg.push_repo_id_adapter)
print(f"Pushed adapter: https://huggingface.co/{cfg.push_repo_id_adapter}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 30.3kB /  646MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpab6yow6b/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Pushed adapter: https://huggingface.co/dizza01/qwen2.5-7b-pdf-lora


In [ ]:
# Load adapter robustly for merge. Some accelerate/device_map combinations can fail with
# TypeError: unhashable type: 'set' in get_balanced_memory.
import json

merge_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

try:
    merge_model = AutoPeftModelForCausalLM.from_pretrained(
        cfg.push_repo_id_adapter,
        dtype=merge_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
    )
except TypeError as exc:
    if "unhashable type: 'set'" not in str(exc):
        raise
    print("[WARN] device-map load path failed; retrying fully on CPU for merge.")
    merge_model = AutoPeftModelForCausalLM.from_pretrained(
        cfg.push_repo_id_adapter,
        dtype=torch.float32,
        device_map=None,
        low_cpu_mem_usage=False,
    )

merge_tokenizer = AutoTokenizer.from_pretrained(cfg.push_repo_id_adapter)

# Endpoint startup fix: some environments expect tokenizer_config.extra_special_tokens to be a dict.
# If it is serialized as a list, sanitize before pushing the merged repo.
tmp_tok_dir = "./_tmp_merged_tokenizer"
os.makedirs(tmp_tok_dir, exist_ok=True)
merge_tokenizer.save_pretrained(tmp_tok_dir)

tok_cfg_path = os.path.join(tmp_tok_dir, "tokenizer_config.json")
if os.path.exists(tok_cfg_path):
    with open(tok_cfg_path, "r", encoding="utf-8") as f:
        tok_cfg = json.load(f)

    extra = tok_cfg.get("extra_special_tokens")
    if isinstance(extra, list):
        tok_cfg["extra_special_tokens"] = {
            f"extra_special_token_{i}": tok
            for i, tok in enumerate(extra)
            if isinstance(tok, str)
        }
        with open(tok_cfg_path, "w", encoding="utf-8") as f:
            json.dump(tok_cfg, f, ensure_ascii=False, indent=2)
        print("[INFO] Sanitized tokenizer_config.extra_special_tokens from list to dict.")

merge_tokenizer_clean = AutoTokenizer.from_pretrained(tmp_tok_dir)

merged_model = merge_model.merge_and_unload()
merged_model.push_to_hub(cfg.push_repo_id_merged)
merge_tokenizer_clean.push_to_hub(cfg.push_repo_id_merged)
print(f"Pushed merged model: https://huggingface.co/{cfg.push_repo_id_merged}")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ssiul3p/model.safetensors:   0%|          | 7.95MB / 15.2GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpd9wt63uw/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed merged model: https://huggingface.co/dizza01/qwen2.5-7b-pdf-merged


In [ ]:
quant_tokenizer = AutoTokenizer.from_pretrained(cfg.push_repo_id_merged)
quant_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

quant_model = AutoModelForCausalLM.from_pretrained(
    cfg.push_repo_id_merged,
    device_map="auto",
    quantization_config=quant_bnb,
)
quant_model.push_to_hub(cfg.push_repo_id_merged_4bit)
quant_tokenizer.push_to_hub(cfg.push_repo_id_merged_4bit)
print(f"Pushed quantized model: https://huggingface.co/{cfg.push_repo_id_merged_4bit}")

Not an apples-to-apples evaluation yet
Your baseline rows are on 900 examples, but finetuned rows are on 200 examples. That alone can shift F1/accuracy materially.
Reference: llm_poc/eval/results/metrics_comparison_plots.ipynb

Training objective mismatch with your target task
The Colab notebook trains on “summarize this excerpt” style text chunks, but your downstream metric is abstention QA behavior. So you optimized for domain writing style, not answer/abstain decision quality.
Reference: llm_poc/model_training/qwen_pdf_finetune_pdf_papers.ipynb

You are likely training on prompt tokens too, not just assistant targets
Current tokenization sets labels = input_ids for the full conversation text. That teaches the model to reproduce system/user prompt text, diluting useful learning signal. For chat SFT, loss should usually be assistant-only.

Data split likely leaks by document
Random chunk split means chunks from the same paper can end up in both train and eval. This gives optimistic training feedback but weak real generalization.

Hyperparameters are aggressive for small, noisy PDF-derived data
LR 2e-4 with r=64 can over-adapt quickly, especially with overlap-heavy chunks and weak supervised targets.

Pipeline bug risk
There is a typo-like artifact before dataclass (/ followed by @dataclass). If that is present when run, config cell is invalid.
Reference: llm_poc/model_training/qwen_pdf_finetune_pdf_papers.ipynb

High-impact improvements for your Colab pipeline:

Train for the actual task
Build supervised QA-abstention examples with explicit target outputs:
If evidence exists: answer succinctly with grounded response
If evidence missing/conflicting: output abstain template
This is the single biggest fix.
Switch to assistant-only loss
Use a chat-template SFT pipeline where only assistant tokens are labeled for loss. Do not compute loss on system/user tokens.

Split by source document, not by chunk
Keep entire papers in either train or eval to prevent leakage.

Make evaluation comparable before conclusions
Re-run finetuned models on 900 examples and same slice/filtering as baselines.

Reduce adaptation strength initially
Try:

learning_rate: 5e-5 or 1e-4
lora_r: 16 or 32
epochs: 1.0 to 2.0 with early stopping
keep one fixed validation set for model selection
Add task-level validation metrics in training loop
Track abstain precision/recall, false answer rate, and F1 on a held-out QA-abstention set each checkpoint. Pick best checkpoint by these metrics, not just LM loss.

Keep merged fp16 as primary deployment candidate first
Evaluate adapter and merged fp16 first. Quantized 4-bit can be published too, but avoid assuming it is equal quality without separate eval.

Improve dataset quality controls
Before training:

Remove boilerplate/references-heavy pages
Deduplicate near-identical chunks
Keep chunk sizes aligned with QA context needs, not only summarization
Add model card metadata at upload
Record base model, data source, intended use, abstention policy, eval table, and known failure modes. This helps reproducibility and avoids confusion between variants.
What I would change first, in order:

Replace chunk-summary training data with QA plus abstain targets
Assistant-only loss
Document-level split
Re-evaluate all models on 900-example abstention set
Then tune LR and LoRA rank
If you want, I can draft a revised Colab cell-by-cell pipeline (drop-in replacement) for:

QA-abstention dataset construction from your papers
assistant-only SFT
checkpoint selection by abstention F1 and false_answer_rate
clean HF upload flow for adapter, merged, and optional 4-bit model.
GPT-5.3-Codex • 1x